# Phân loại CIFAR-100 bằng ResNet18 và Swin-T

**CO5177 · Image · Nhóm CTG**  
Huỳnh Thành Đạt (2570161) và Lương Minh Duy (2570083).

Notebook thực hiện một lượt nghiên cứu đầy đủ: khám phá CIFAR-100, huấn luyện các cấu hình pretrained, chọn checkpoint theo validation, đánh giá test sau khi cố định giao thức, rồi tổng hợp metric, đường cong và confusion matrix. Mỗi lần chạy tạo thư mục kết quả mới; không đọc các prediction hay metric CO5085 làm kết quả hiện tại. Kết quả CO5085 được giữ riêng như phép đo lịch sử có cùng nguồn dữ liệu và giao thức.

Ảnh CIFAR-100 gốc có kích thước 32×32. Đầu vào 96×96 được giữ để đối chiếu với protocol CO5085; nội suy không tạo thêm thông tin ảnh và vẫn thấp hơn mức cạnh ngắn 128 px được khuyến nghị trong đề. Sáu cấu hình pretrained là so sánh chính.


## 1. Cấu hình và môi trường

Mặc định Run all chạy đủ sáu cấu hình pretrained với seed 42, 43 và 44 (18 lượt, 20 epoch mỗi lượt). Có thể đặt `CO5177_IMAGE_SEEDS=42` để chạy shard seed đơn; đặt `CO5177_IMAGE_OUTPUT_ROOT=/content/image-runs/shard-42` để chọn đường dẫn đầu ra cố định (thư mục phải chưa tồn tại). `CO5177_IMAGE_DATA_ROOT` trỏ tới cache CIFAR-100 đã tải và được torchvision xác thực. Mỗi shard cần thư mục output riêng. Không chạy lại vào thư mục đã có kết quả. Các cấu hình scratch trong protocol không thuộc lượt mặc định; có thể thêm ID vào `EXPERIMENT_IDS` như một phân tích ngữ cảnh riêng.

Giao thức yêu cầu GPU hỗ trợ BF16. Cell kiểm tra dừng với thông báo nếu runtime không phù hợp; không tự đổi precision hay CPU/GPU. CIFAR-100 và trọng số torchvision pretrained được tải vào cache của runtime, không ghi vào Git.


In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util, json, time, hashlib, contextlib, csv, copy
from datetime import datetime, timezone
import platform, importlib.metadata

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)
if not Path('configs/image.json').is_file():
    repo = Path('/content/CO5177-PFDAV')
    if not repo.exists():
        subprocess.run(['git', 'clone', 'https://github.com/dathuynh1108/CO5177-PFDAV.git', str(repo)], check=True)
    os.chdir(repo)
missing = [name for name in ['numpy','matplotlib','PIL','torch','torchvision'] if importlib.util.find_spec(name) is None]
if missing:
    package_names = ['pillow' if n == 'PIL' else n for n in missing]
    subprocess.run([sys.executable, '-m', 'pip', 'install', *package_names], check=True)
sys.path.insert(0, str(Path('src').resolve()))
import numpy as np
import matplotlib.pyplot as plt
import torch, torchvision
from torch.utils.data import Subset
from torchvision.datasets import CIFAR100
from IPython.display import Image, display
from image.protocol import load_protocol
from image.data import EpochSubset, stratified_split, transform, loader
from image.engine import epoch_pass, better
from image.models import build_model, frozen_state
from image.runtime import choose_device, seed_everything, atomic_json, atomic_csv, atomic_torch_save, state_fingerprint, environment_info

# Chỉnh các giá trị này trước khi Run all. Mỗi shard dùng seed khác và thư mục mới.
EXPERIMENT_IDS = ['resnet18_head','resnet18_partial','resnet18_full',
                  'swin_t_head','swin_t_partial','swin_t_full']
SEEDS = [int(value.strip()) for value in os.environ.get('CO5177_IMAGE_SEEDS','42,43,44').split(',') if value.strip()]
WORKERS = min(4, os.cpu_count() or 1)
DEVICE_REQUEST = 'cuda'
DEFAULT_DATA_ROOT = Path('/content/cifar100') if Path('/content').exists() else Path('/tmp/cifar100')
DATA_ROOT = Path(os.environ.get('CO5177_IMAGE_DATA_ROOT', str(DEFAULT_DATA_ROOT))).expanduser().resolve()
RUN_ROOT = Path('/content/image-runs') if Path('/content').exists() else Path('/tmp/image-runs')
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
FIXED_OUTPUT_ROOT = os.environ.get('CO5177_IMAGE_OUTPUT_ROOT')
OUTPUT_ROOT = Path(FIXED_OUTPUT_ROOT).expanduser().resolve() if FIXED_OUTPUT_ROOT else (RUN_ROOT / f'co5177-image-{STAMP}').resolve()
PROTOCOL = load_protocol()
DEVICE = choose_device(DEVICE_REQUEST)
assert DEVICE.type == 'cuda', 'Chọn runtime có GPU CUDA để chạy protocol Image.'
assert torch.cuda.is_bf16_supported(), 'Protocol yêu cầu GPU CUDA hỗ trợ BF16 (ví dụ L4, A100 hoặc H100).'
assert SEEDS and len(SEEDS) == len(set(SEEDS)) and all(isinstance(s, int) and s >= 0 for s in SEEDS)
all_experiments = {e['id']: e for e in PROTOCOL['experiments']}
assert len(EXPERIMENT_IDS) == len(set(EXPERIMENT_IDS)) and set(EXPERIMENT_IDS) <= all_experiments.keys()
EXPERIMENTS = [all_experiments[k] for k in EXPERIMENT_IDS]
OUTPUT_ROOT.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(exist_ok=False)  # Fixed output roots are never overwritten.
print(f'Repository: {Path.cwd()}')
print(f'Python {platform.python_version()} · PyTorch {torch.__version__} · torchvision {torchvision.__version__}')
print(f'NumPy {np.__version__} · Matplotlib {plt.matplotlib.__version__} · Pillow {importlib.metadata.version("pillow")}')
print(f'CUDA {torch.version.cuda} · thiết bị {torch.cuda.get_device_name(DEVICE)} · BF16 native={torch.cuda.is_bf16_supported()}')
print(f'DATA_ROOT={DATA_ROOT} · output root cố định={bool(FIXED_OUTPUT_ROOT)}')
print(f'Protocol: {len(EXPERIMENTS)} cấu hình × {len(SEEDS)} seed × {PROTOCOL["epochs"]} epoch')
print(f'Thư mục kết quả mới: {OUTPUT_ROOT}')

def show_figure(figure, name):
    path = OUTPUT_ROOT / name
    figure.savefig(path, dpi=150, bbox_inches='tight')
    display(Image(filename=str(path)))
    plt.close(figure)


## 2. Tải dữ liệu và khám phá

Tải hai partition CIFAR-100 chính thức bằng torchvision. Tập train chính thức được chia phân tầng theo seed 2026 thành train và validation. Trước huấn luyện, audit SHA256 trên pixel RGB giữ một đại diện cho mỗi ảnh trùng chính xác với thứ tự ưu tiên test, validation, train; vì vậy kích thước cuối có thể nhỏ hơn 45.000/5.000/10.000. Không đánh giá near-duplicates. Dữ liệu split và chỉ số bị loại được lưu để kiểm tra. Biểu đồ dưới đây dựng từ dữ liệu vừa tải, không dùng ảnh hoặc kết quả xuất sẵn.


In [ ]:
# Ưu tiên endpoint Toronto của torchvision. Khi tải thất bại, dùng mirror tar có MD5
# chính thức; torchvision tự xác thực archive và từng batch dữ liệu trích xuất.
DATA_ROOT.mkdir(parents=True,exist_ok=True)
class CIFAR100ChecksumMirror(CIFAR100):
    url='https://data.brainchip.com/dataset-mirror/cifar100/cifar-100-python.tar.gz'
    filename='cifar-100-python.tar.gz'
    tgz_md5='eb9058c3a382ffc7106e4002c42a8d85'

try:
    train_raw=CIFAR100(str(DATA_ROOT),train=True,download=True)
    test_raw=CIFAR100(str(DATA_ROOT),train=False,download=True)
    dataset_download_source='University of Toronto (torchvision official URL)'
except Exception as primary_error:
    print(f'Tải từ Toronto thất bại: {type(primary_error).__name__}; thử mirror CIFAR-100 đã xác thực MD5.')
    try:
        CIFAR100ChecksumMirror(str(DATA_ROOT),train=True,download=True)
        CIFAR100ChecksumMirror(str(DATA_ROOT),train=False,download=True)
        train_raw=CIFAR100(str(DATA_ROOT),train=True,download=False)
        test_raw=CIFAR100(str(DATA_ROOT),train=False,download=False)
        dataset_download_source='BrainChip CIFAR mirror; archive MD5 verified against torchvision official digest'
    except Exception as mirror_error:
        raise RuntimeError(f'Cả nguồn Toronto và mirror CIFAR-100 đều không dùng được: {mirror_error}') from primary_error
print('Nguồn tải dữ liệu:',dataset_download_source)
train_indices, val_indices = stratified_split(train_raw.targets, PROTOCOL['split_seed'])

def pixel_hash(image):
    return hashlib.sha256(np.ascontiguousarray(image).tobytes()).hexdigest()

train_hashes = [pixel_hash(image) for image in train_raw.data]
test_hashes = [pixel_hash(image) for image in test_raw.data]
# Ưu tiên giữ bản test chính thức; loại nội dung trùng khỏi validation/train.
# Trong từng tập, giữ một đại diện cho mỗi nội dung RGB chính xác.
def unique_indices(indices, hashes, excluded=frozenset()):
    kept, seen = [], set(excluded)
    for index in indices:
        key = hashes[index]
        if key not in seen:
            kept.append(index); seen.add(key)
    return kept, seen

clean_test_indices, test_hash_set = unique_indices(range(len(test_hashes)), test_hashes)
clean_val_indices, val_hash_set = unique_indices(val_indices, train_hashes, test_hash_set)
clean_train_indices, train_hash_set = unique_indices(train_indices, train_hashes, val_hash_set)
assert not (set(train_hashes[i] for i in clean_train_indices) & set(train_hashes[i] for i in clean_val_indices))
assert not (set(train_hashes[i] for i in clean_train_indices) & set(test_hashes[i] for i in clean_test_indices))
assert not (set(train_hashes[i] for i in clean_val_indices) & set(test_hashes[i] for i in clean_test_indices))
assert len(clean_test_indices) > 0
# Lớp ảnh trùng được giữ theo ưu tiên test > validation > train; lưu danh sách chỉ số.
split_audit = {
    'policy':'SHA256 exact RGB-pixel deduplication; test representative has priority, then validation, then train; one representative per exact image content per split',
    'split_seed':PROTOCOL['split_seed'], 'official_development_train':len(train_indices),
    'official_development_validation':len(val_indices), 'official_test':len(test_hashes),
    'retained_train':len(clean_train_indices), 'retained_validation':len(clean_val_indices),
    'retained_test':len(clean_test_indices), 'removed_train':len(train_indices)-len(clean_train_indices),
    'removed_validation':len(val_indices)-len(clean_val_indices), 'removed_test':len(test_hashes)-len(clean_test_indices),
    'train_indices':clean_train_indices, 'validation_indices':clean_val_indices, 'test_indices':clean_test_indices,
    'duplicates_considered_exact_pixels_only':True,
}
class_names = train_raw.classes
train_counts=np.bincount(np.asarray(train_raw.targets)[clean_train_indices],minlength=100)
fig,ax=plt.subplots(figsize=(14,4)); ax.bar(np.arange(100),train_counts,color='#49685a')
ax.set(xlabel='Fine class ID',ylabel='Retained training images',title='Phân bố lớp sau khi loại ảnh trùng chính xác')
ax.set_xticks(np.arange(0,100,5)); ax.grid(axis='y',alpha=.2); fig.tight_layout(); show_figure(fig, "class_distribution.png")
fig,axes=plt.subplots(2,5,figsize=(12,5))
for ax,index in zip(axes.flat,clean_train_indices[:10]):
    ax.imshow(train_raw.data[index]); ax.set_title(class_names[int(train_raw.targets[index])],fontsize=9); ax.axis('off')
fig.suptitle('Mẫu ảnh train CIFAR-100 (32×32)'); fig.tight_layout(); show_figure(fig, "training_examples.png")
print(f"Số lớp: 100 · ảnh RGB 32×32 · train/validation/test sau dedup: {len(clean_train_indices):,}/{len(clean_val_indices):,}/{len(clean_test_indices):,}")
print('Ảnh bị loại khỏi các split ban đầu:', split_audit['removed_train'], 'train,', split_audit['removed_validation'], 'validation,', split_audit['removed_test'], 'test')


## 3. Huấn luyện và chọn checkpoint

Mỗi lượt dùng augmentation chỉ trên train, cùng normalization ImageNet và đầu vào 96×96. Mỗi epoch ghi accuracy, macro-F1, cross-entropy và thời gian cho train/validation. Checkpoint được chọn theo validation accuracy; khi hòa, chọn validation cross-entropy thấp nhất. Mỗi lượt có checkpoint, history CSV, metadata giao thức, seed, môi trường và log riêng. Để áp dụng split đã loại trùng mà không sửa nguồn đo CO5085, notebook gọi trực tiếp các hàm low-level `epoch_pass`, `EpochSubset`, `loader` và `build_model`; dữ liệu huấn luyện chỉ chứa chỉ số đã qua audit.

Cell dưới gọi trực tiếp hàm `train_run` của training engine trong notebook; không gọi subprocess/CLI và không nạp checkpoint CO5085.


In [ ]:
from image.runtime import atomic_json, atomic_csv, atomic_torch_save, state_fingerprint, environment_info, synchronize
from image.data import development_data
from image.protocol import CONFIG

# Gói dataset đúng với policy lọc trùng và transform trong protocol.
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
train_base=CIFAR100(str(DATA_ROOT),train=True,download=False,transform=transform(PROTOCOL,train=True))
val_base=CIFAR100(str(DATA_ROOT),train=True,download=False,transform=transform(PROTOCOL))
test_base=CIFAR100(str(DATA_ROOT),train=False,download=False,transform=transform(PROTOCOL))
train_datasets={}; val_datasets={}; test_datasets={}
for seed in SEEDS:
    train_datasets[seed]=EpochSubset(train_base,clean_train_indices)
    val_datasets[seed]=Subset(val_base,clean_val_indices)
    test_datasets[seed]=Subset(test_base,clean_test_indices)

notebook_path=Path('notebooks/image.ipynb')
notebook_bytes=notebook_path.read_bytes() if notebook_path.is_file() else b''
notebook_source_sha256=hashlib.sha256(notebook_bytes).hexdigest() if notebook_bytes else None
notebook_code_digest=hashlib.sha256('\n'.join(''.join(c.get('source',[])) for c in json.loads(notebook_bytes).get('cells',[]) if c.get('cell_type')=='code').encode()).hexdigest() if notebook_bytes else None
dependency_versions={'python':platform.python_version(),'torch':torch.__version__,'torchvision':torchvision.__version__,
                     'numpy':np.__version__,'matplotlib':plt.matplotlib.__version__,
                     'pillow':importlib.metadata.version('pillow'),'cuda':torch.version.cuda,
                     'bf16_native':torch.cuda.is_bf16_supported()}
config_snapshot={
    'created_utc':datetime.now(timezone.utc).isoformat(),'protocol':PROTOCOL,
    'experiment_ids':EXPERIMENT_IDS,'seeds':SEEDS,'workers':WORKERS,'device':str(DEVICE),
    'data_root':str(DATA_ROOT),'output_root':str(OUTPUT_ROOT),'split_audit':split_audit,
    'pixel_sha256_train_ordered':hashlib.sha256(''.join(train_hashes).encode()).hexdigest(),
    'pixel_sha256_test_ordered':hashlib.sha256(''.join(test_hashes).encode()).hexdigest(),
    'source':'notebook visible loop using image.engine.epoch_pass and image.data loader/datasets',
    'notebook_source_sha256':notebook_source_sha256,'notebook_code_digest':notebook_code_digest,
    'dependency_versions':dependency_versions,'declared_direct_requirements':{'torch':'2.10.0','torchvision':'0.25.0','numpy':'>=1.26,<3','matplotlib':'>=3.8,<4','pillow':'>=10,<13'},'dataset_download_source':dataset_download_source,
}
atomic_json(OUTPUT_ROOT/'notebook_config.json',config_snapshot)
atomic_json(OUTPUT_ROOT/'dedup_split.json',split_audit)
train_metadata={}; histories={}
for experiment in EXPERIMENTS:
  for seed in SEEDS:
    run_dir=OUTPUT_ROOT/experiment['id']/f'seed_{seed}'
    run_dir.mkdir(parents=True,exist_ok=False)
    log_path=run_dir/'notebook.log'; history=[]; seed_everything(seed)
    model=build_model(experiment,initialize_pretrained=True).to(DEVICE)
    optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=experiment['lr'],
        weight_decay=PROTOCOL['weight_decay'],betas=tuple(PROTOCOL['betas']),eps=PROTOCOL['eps'])
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=PROTOCOL['epochs'])
    train_loader=loader(train_datasets[seed],PROTOCOL['batch_size'],WORKERS,torch.Generator().manual_seed(seed),True,DEVICE)
    val_loader=loader(val_datasets[seed],PROTOCOL['batch_size'],WORKERS,torch.Generator().manual_seed(seed+1),False,DEVICE)
    best=None; frozen_hash=state_fingerprint(frozen_state(model)); train_seconds_total=0.0
    for epoch in range(1,PROTOCOL['epochs']+1):
      train_datasets[seed].set_epoch(epoch,seed)
      if DEVICE.type=='cuda': torch.cuda.reset_peak_memory_stats(DEVICE)
      synchronize(DEVICE); t0=time.perf_counter(); lr=optimizer.param_groups[0]['lr']
      train_metric=epoch_pass(model,train_loader,DEVICE,optimizer,PROTOCOL['gradient_clip'])
      synchronize(DEVICE); train_seconds=time.perf_counter()-t0; train_seconds_total+=train_seconds
      val_metric=epoch_pass(model,val_loader,DEVICE); synchronize(DEVICE)
      epoch_seconds=time.perf_counter()-t0
      assert state_fingerprint(frozen_state(model))==frozen_hash, 'Frozen backbone changed during training.'
      row={'epoch':epoch,'lr':lr,'train_ce':train_metric['ce'],'train_accuracy':train_metric['accuracy'],
           'train_macro_f1':train_metric['macro_f1'],'val_ce':val_metric['ce'],'val_accuracy':val_metric['accuracy'],
           'val_macro_f1':val_metric['macro_f1'],'train_seconds':train_seconds,
           'epoch_seconds':epoch_seconds,'peak_gpu_memory_bytes':torch.cuda.max_memory_allocated(DEVICE) if DEVICE.type=='cuda' else 0}
      if better(val_metric['accuracy'],val_metric['ce'],best):
        best={'epoch':epoch,'accuracy':val_metric['accuracy'],'ce':val_metric['ce'],'macro_f1':val_metric['macro_f1'],
              'state_sha256':state_fingerprint(model.state_dict())}
        atomic_torch_save(run_dir/'best.pt',{'model':model.state_dict(),'selection':best,'seed':seed,'experiment':experiment['id']})
      history.append(row); scheduler.step()
      with log_path.open('a',encoding='utf-8') as log: log.write(f"epoch={epoch}/{PROTOCOL['epochs']} train_acc={train_metric['accuracy']:.6f} val_acc={val_metric['accuracy']:.6f} val_ce={val_metric['ce']:.6f} seconds={epoch_seconds:.1f}\n")
      atomic_csv(run_dir/'history.csv',history)
      print(f"{experiment['id']} seed={seed} epoch={epoch}/{PROTOCOL['epochs']} val_acc={val_metric['accuracy']:.4f}")
    metadata={'experiment':experiment,'seed':seed,'device':str(DEVICE),'environment':environment_info(DEVICE),
              'completed_epochs':len(history),'status':'trained','best_validation':best,'train_seconds':train_seconds_total,
              'epoch_seconds':sum(r['epoch_seconds'] for r in history),'peak_gpu_memory_bytes':max(r['peak_gpu_memory_bytes'] for r in history),
              'retained_train':len(clean_train_indices),'retained_validation':len(clean_val_indices),'test_evaluated':False,
              'selection':'max validation accuracy; tie min validation CE','split_policy':split_audit['policy']}
    atomic_json(run_dir/'metadata.json',metadata)
    histories[(experiment['id'],seed)]=history; train_metadata[(experiment['id'],seed)]=metadata
    del optimizer,scheduler,model,train_loader,val_loader
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
atomic_json(OUTPUT_ROOT/'training_summary.json',{'runs':[{'experiment':k[0],'seed':k[1],**{f:v for f,v in m.items() if f in ['status','completed_epochs','best_validation','train_seconds','epoch_seconds']}} for k,m in train_metadata.items()]})
print('Training xong; đã lưu history, best checkpoint, cấu hình và log. Chưa mở kết quả test.')


## 4. Đánh giá trên test sau khi cố định protocol

Các cấu hình, seed, preprocessing và tiêu chí chọn checkpoint đã được ghi trong `notebook_config.json` trước khi đánh giá test. Cell đánh giá nạp checkpoint tốt nhất theo validation của chính lượt chạy mới, ghi prediction/metric/confusion matrix và không thay đổi mô hình.


In [ ]:
# Test dùng các split đã audit, chỉ sau khi cấu hình, seed và tiêu chí chọn checkpoint đã được ghi.
test_results={}
for experiment in EXPERIMENTS:
  for seed in SEEDS:
    run_dir=OUTPUT_ROOT/experiment['id']/f'seed_{seed}'
    checkpoint=torch.load(run_dir/'best.pt',map_location='cpu',weights_only=False)
    model=build_model(experiment,initialize_pretrained=False).to(DEVICE)
    model.load_state_dict(checkpoint['model']); assert state_fingerprint(model.state_dict())==checkpoint['selection']['state_sha256']
    test_loader=loader(test_datasets[seed],PROTOCOL['batch_size'],WORKERS,torch.Generator().manual_seed(seed+2),False,DEVICE)
    evaluation=epoch_pass(model,test_loader,DEVICE,predictions=True)
    predictions=evaluation.pop('predictions')
    for prediction in predictions:
        prediction['official_index'] = clean_test_indices[prediction['index']]
    for row in evaluation['per_class']: row['class_name']=class_names[row['class_id']]
    evaluation.update({'selected_epoch':checkpoint['selection']['epoch'],'selection':checkpoint['selection'],
                       'evaluation_count':len(clean_test_indices),'split_policy':split_audit['policy']})
    atomic_json(run_dir/'evaluation_test.json',evaluation)
    atomic_csv(run_dir/'predictions_test.csv',predictions)
    atomic_csv(run_dir/'per_class_test.csv',evaluation['per_class'])
    atomic_csv(run_dir/'confusion_test.csv',[{'true_class':i,**{str(j):n for j,n in enumerate(row)}} for i,row in enumerate(evaluation['confusion'])])
    metadata=json.loads((run_dir/'metadata.json').read_text()); metadata['test_evaluated']=True; atomic_json(run_dir/'metadata.json',metadata)
    test_results[(experiment['id'],seed)]=evaluation
    print(f"{experiment['id']} seed={seed}: test acc={evaluation['accuracy']:.4f}, macro-F1={evaluation['macro_f1']:.4f}, n={len(clean_test_indices)}")
    del model,checkpoint,test_loader
    if DEVICE.type=='cuda': torch.cuda.empty_cache()


## 5. Metric, chi phí và biểu đồ

Bảng và biểu đồ được tính từ `evaluation_test.json` và `history.csv` trong thư mục lượt chạy vừa tạo. Giá trị tổng hợp là trung bình và độ lệch chuẩn mẫu qua các seed được chọn.


In [ ]:
summary_rows = []
for experiment in EXPERIMENTS:
    runs = [(seed, test_results[(experiment['id'],seed)], train_metadata[(experiment['id'],seed)]) for seed in SEEDS]
    acc = np.asarray([r[1]['accuracy'] for r in runs], dtype=float)
    f1 = np.asarray([r[1]['macro_f1'] for r in runs], dtype=float)
    train_sec = np.asarray([r[2]['train_seconds'] for r in runs], dtype=float)
    peak_mib = np.asarray([r[2]['peak_gpu_memory_bytes'] / 1024**2 for r in runs], dtype=float)
    summary_rows.append({'experiment':experiment['id'], 'accuracy_mean':float(acc.mean()),
                         'accuracy_sd':float(acc.std(ddof=1)) if len(acc)>1 else 0.0,
                         'macro_f1_mean':float(f1.mean()), 'macro_f1_sd':float(f1.std(ddof=1)) if len(f1)>1 else 0.0,
                         'train_seconds_mean':float(train_sec.mean()), 'peak_memory_mib_mean':float(peak_mib.mean())})
with (OUTPUT_ROOT/'metrics_summary.csv').open('w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f,fieldnames=list(summary_rows[0])); writer.writeheader(); writer.writerows(summary_rows)
fig, ax = plt.subplots(figsize=(11,4.8)); x=np.arange(len(summary_rows))
means=[100*r['accuracy_mean'] for r in summary_rows]; errs=[100*r['accuracy_sd'] for r in summary_rows]
ax.bar(x,means,yerr=errs,capsize=4,color='#49685a'); ax.set_xticks(x,[r['experiment'].replace('_','\n',1) for r in summary_rows],rotation=15,ha='right')
ax.set(ylabel='Test accuracy (%)',title='Kết quả test từ các lượt huấn luyện mới'); ax.grid(axis='y',alpha=.2); fig.tight_layout(); show_figure(fig, "test_accuracy.png")
for row in summary_rows:
    print(f"{row['experiment']:20s} acc={100*row['accuracy_mean']:.2f} ± {100*row['accuracy_sd']:.2f}%  "
          f"macro-F1={row['macro_f1_mean']:.4f} ± {row['macro_f1_sd']:.4f}  "
          f"train={row['train_seconds_mean']/60:.1f} min  peak={row['peak_memory_mib_mean']:.0f} MiB")


## 6. Đường cong học và confusion matrix

Đường cong lấy từ history CSV mới. Confusion matrix hiển thị hai cấu hình full fine-tuning; ma trận và prediction đầy đủ của mọi cấu hình được lưu theo từng run.


In [ ]:
fig,axes=plt.subplots(len(EXPERIMENTS),2,figsize=(11,3.0*len(EXPERIMENTS)),squeeze=False)
for row_idx,experiment in enumerate(EXPERIMENTS):
  for ax,metric in zip(axes[row_idx],['accuracy','ce']):
    for split_name in ['train','val']:
      values=np.stack([[row[f'{split_name}_{metric}'] for row in histories[(experiment['id'],seed)]] for seed in SEEDS])
      mean=values.mean(0); sd=values.std(0,ddof=1) if len(values)>1 else np.zeros_like(mean); epoch=np.arange(1,len(mean)+1)
      ax.plot(epoch,mean,label=split_name); ax.fill_between(epoch,mean-sd,mean+sd,alpha=.15)
    ax.set(title=f"{experiment['id']} · {metric}",xlabel='Epoch',ylabel=metric); ax.grid(alpha=.2); ax.legend()
fig.tight_layout(); show_figure(fig, "learning_curves.png")
for experiment_id in [i for i in ['resnet18_full','swin_t_full'] if i in EXPERIMENT_IDS]:
  confusions=[np.asarray(test_results[(experiment_id,s)]['confusion']) for s in SEEDS]
  cm=np.rint(np.mean(confusions,axis=0)).astype(int)
  fig,ax=plt.subplots(figsize=(8,7)); im=ax.imshow(cm,cmap='Blues',interpolation='nearest')
  ax.set(title=f'{experiment_id}: confusion matrix trung bình qua seed',xlabel='Nhãn dự đoán',ylabel='Nhãn thật')
  fig.colorbar(im,ax=ax,label='Số ảnh (trung bình làm tròn)'); fig.tight_layout(); show_figure(fig, f"{experiment_id}-confusion.png")


## 7. Kết luận và tệp đầu ra

Notebook đã chạy huấn luyện mới nếu các cell trên hoàn tất. `OUTPUT_ROOT` chứa cấu hình, seed, log console, lịch sử epoch, metadata/chữ ký, checkpoint tốt nhất, dự đoán test, metric từng lớp, confusion matrix và các hình được dựng từ chính lượt chạy này. Tệp cũ tại `reports/image/generated/evidence/` không được dùng làm đầu vào cho các bảng hoặc đồ thị ở trên.

Nội dung sử dụng AI được ghi tại [AI_USE_REPORT.md](https://github.com/dathuynh1108/CO5177-PFDAV/blob/main/AI_USE_REPORT.md).
